In [4]:
from pathlib import Path
import pandas as pd

In [23]:
DATA_DIR = Path("data")

for file in DATA_DIR.glob("*.csv"):
    df = pd.read_csv(file)

    print("\n" + "=" * 50)
    print("FILE:", file.name)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Column names:", df.columns.tolist())

    print("\nFirst 3 records:")
    print(df.head(3).to_string(index=False))

    print("\nMissing values:")
    print(df.isna().sum().to_string())

    print("Duplicate rows:", df.duplicated().sum())



FILE: data_dictionary.csv
Rows: 71
Columns: 4
Column names: ['table_name', 'column_name', 'data_type', 'description']

First 3 records:
 table_name       column_name data_type                                                           description
departments     department_id    object     Field department_id in departments; see README for table meaning.
departments   department_name    object   Field department_name in departments; see README for table meaning.
departments business_function    object Field business_function in departments; see README for table meaning.

Missing values:
table_name     0
column_name    0
data_type      0
description    0
Duplicate rows: 0

FILE: departments.csv
Rows: 8
Columns: 3
Column names: ['department_id', 'department_name', 'business_function']

First 3 records:
department_id department_name    business_function
         D001           Sales     Revenue & Growth
         D002       Marketing     Revenue & Growth
         D003     Engineering Produ

In [29]:
# WORKFORCEIQ: DATA QUALITY TESTS

# Test 1: Employee manager IDs must reference real employees
employee_ids = set(employees["employee_id"])

invalid_managers = employees[
    ~employees["manager_employee_id"].isin(employee_ids)
]

print("Invalid manager references:", len(invalid_managers))

# Test 2: Employee department must match their department ID
department_lookup = departments.set_index(
    "department_id"
)["department_name"]

expected_department = employees["department_id"].map(
    department_lookup
)

department_mismatch = employees[
    employees["department_name"] != expected_department
]

print("Employee department mismatches:", len(department_mismatch))

# Test 3: Employee role must belong to their department
role_lookup = roles.set_index("role_id")["department_name"]

expected_role_department = employees["role_id"].map(role_lookup)

role_mismatch = employees[
    employees["department_name"] != expected_role_department
]

print("Employee role/department mismatches:", len(role_mismatch))

# Test 4: Recalculate KPI achievement
higher = performance_records["better_direction"] == "higher"
lower = performance_records["better_direction"] == "lower"

calculated = pd.Series(
    0.0, index=performance_records.index
)

calculated.loc[higher] = (
    performance_records.loc[higher, "actual_value"]
    / performance_records.loc[higher, "target_value"]
    * 100
)

calculated.loc[lower] = (
    performance_records.loc[lower, "target_value"]
    / performance_records.loc[lower, "actual_value"].replace(0, float("nan"))
    * 100
)

calculated = calculated.fillna(150).clip(upper=150)

score_difference = (
    calculated - performance_records["achievement_pct"]
).abs()

print(
    "KPI score mismatches:",
    int((score_difference > 0.02).sum())
)

# Test 5: Every performance record must use
# the department assigned to its employee
employee_department_lookup = employees.set_index(
    "employee_id"
)["department_id"]

expected_record_department = (
    performance_records["employee_id"]
    .map(employee_department_lookup)
)

record_department_mismatch = (
    performance_records["department_id"]
    != expected_record_department
)

print(
    "Performance department mismatches:",
    int(record_department_mismatch.sum())
)

# Test 6: Inspect pipeline outcomes
print("\nPipeline run counts:")
print(pipeline_runs["status"].value_counts())

print("\nAll tests finished.")


Invalid manager references: 0
Employee department mismatches: 0
Employee role/department mismatches: 0
KPI score mismatches: 8230
Performance department mismatches: 0

Pipeline run counts:
status
Success    159
Failed      21
Name: count, dtype: int64

All tests finished.


In [31]:
employees = pd.read_csv(DATA_DIR / "employees.csv")
departments = pd.read_csv(DATA_DIR / "departments.csv")
roles = pd.read_csv(DATA_DIR / "roles.csv")
kpi_definitions = pd.read_csv(DATA_DIR / "kpi_definitions.csv")
kpi_targets = pd.read_csv(DATA_DIR / "kpi_targets.csv")
performance_records = pd.read_csv(
    DATA_DIR / "performance_records.csv"
)
pipeline_runs = pd.read_csv(DATA_DIR / "pipeline_runs.csv")

print("All required tables loaded successfully.")
print("Employees:", len(employees))
print("Performance records:", len(performance_records))


All required tables loaded successfully.
Employees: 720
Performance records: 40250


In [35]:

# DIAGNOSE KPI SCORE MISMATCHES

diagnostic = performance_records.copy()

higher = diagnostic["better_direction"] == "higher"
lower = diagnostic["better_direction"] == "lower"

diagnostic["recalculated_score"] = 0.0

diagnostic.loc[higher, "recalculated_score"] = (
    diagnostic.loc[higher, "actual_value"]
    / diagnostic.loc[higher, "target_value"]
    * 100
)

nonzero_lower = lower & (diagnostic["actual_value"] > 0)
zero_lower = lower & (diagnostic["actual_value"] == 0)

diagnostic.loc[nonzero_lower, "recalculated_score"] = (
    diagnostic.loc[nonzero_lower, "target_value"]
    / diagnostic.loc[nonzero_lower, "actual_value"]
    * 100
)

diagnostic.loc[zero_lower, "recalculated_score"] = 150

diagnostic["recalculated_score"] = (
    diagnostic["recalculated_score"].clip(upper=150).round(2)
)

diagnostic["score_difference"] = (
    diagnostic["recalculated_score"]
    - diagnostic["achievement_pct"]
).abs()

print("Mismatch count by KPI unit:")
display(
    diagnostic.groupby("unit")
    .agg(
        records=("unit", "size"),
        mismatches=("score_difference", lambda x: (x > 0.02).sum()),
        average_difference=("score_difference", "mean"),
        maximum_difference=("score_difference", "max"),
    )
    .sort_values("mismatches", ascending=False)
)

print("\nExample mismatched records:")
display(
    diagnostic.loc[
        diagnostic["score_difference"] > 0.02,
        [
            "employee_id",
            "kpi_id",
            "reporting_month",
            "unit",
            "target_value",
            "actual_value",
            "achievement_pct",
            "recalculated_score",
            "score_difference",
        ],
    ].head(15)
)


Mismatch count by KPI unit:


,records,mismatches,average_difference,maximum_difference
unit,,,,
percent,21809,3315,0.012930,0.31
hours,4212,1600,0.022355,0.13
currency_usd,878,764,0.102369,0.43
ratio,865,750,0.081630,0.17
days,2424,667,0.018647,0.16
count,5392,529,0.006200,0.05
minutes,1173,200,0.012651,0.04
score_100,3497,0,0.002316,0.01



Example mismatched records:


,employee_id,kpi_id,reporting_month,unit,target_value,actual_value,achievement_pct,recalculated_score,score_difference
66,E0002,K012,2025-02,percent,3.97,7.07,56.12,56.15,0.03
69,E0002,K015,2025-02,hours,7.31,12.63,57.86,57.88,0.02
71,E0002,K012,2025-03,percent,3.97,5.62,70.61,70.64,0.03
79,E0002,K015,2025-04,hours,7.31,12.05,60.68,60.66,0.02
81,E0002,K012,2025-05,percent,3.97,5.09,77.96,78.00,0.04
86,E0002,K012,2025-06,percent,3.97,6.40,62.01,62.03,0.02
91,E0002,K012,2025-07,percent,3.97,6.17,64.39,64.34,0.05
94,E0002,K015,2025-07,hours,7.31,8.91,82.01,82.04,0.03
99,E0002,K015,2025-08,hours,7.31,12.40,58.93,58.95,0.02
104,E0002,K015,2025-09,hours,7.31,10.17,71.84,71.88,0.04


In [37]:

tolerances = [0.01, 0.02, 0.05, 0.10, 0.25, 0.50]

tolerance_results = []

for tolerance in tolerances:
    count = int(
        (diagnostic["score_difference"] > tolerance).sum()
    )

    tolerance_results.append({
        "tolerance_percentage_points": tolerance,
        "mismatch_count": count,
        "mismatch_rate_pct": round(
            count / len(diagnostic) * 100, 2
        ),
    })

display(pd.DataFrame(tolerance_results))

print("Largest discrepancies:")
display(
    diagnostic.nlargest(10, "score_difference")[
        [
            "employee_id",
            "kpi_id",
            "reporting_month",
            "unit",
            "target_value",
            "actual_value",
            "achievement_pct",
            "recalculated_score",
            "score_difference",
        ]
    ]
)


,tolerance_percentage_points,mismatch_count,mismatch_rate_pct
0,0.01,15117,37.56
1,0.02,7825,19.44
2,0.05,3833,9.52
3,0.10,1407,3.50
4,0.25,32,0.08
5,0.50,0,0.00


Largest discrepancies:


,employee_id,kpi_id,reporting_month,unit,target_value,actual_value,achievement_pct,recalculated_score,score_difference
13863,E0246,K034,2025-08,currency_usd,2.31,1.58,145.77,146.20,0.43
3928,E0070,K034,2025-04,currency_usd,2.31,1.69,136.28,136.69,0.41
13458,E0239,K034,2025-05,currency_usd,2.31,1.65,140.39,140.00,0.39
17963,E0319,K034,2025-10,currency_usd,2.31,1.76,131.62,131.25,0.37
17923,E0319,K034,2025-02,currency_usd,2.31,1.80,127.98,128.33,0.35
17183,E0305,K034,2025-06,currency_usd,2.31,1.84,125.22,125.54,0.32
9073,E0160,K039,2025-08,percent,3.29,2.25,145.91,146.22,0.31
16918,E0301,K034,2025-01,currency_usd,2.31,1.90,121.89,121.58,0.31
32748,E0586,K034,2025-02,currency_usd,2.31,1.87,123.84,123.53,0.31
16073,E0285,K034,2025-10,currency_usd,2.63,1.98,133.14,132.83,0.31


In [39]:

# Inspect the KPI definition and its scoring direction

display(
    kpi_definitions.loc[
        kpi_definitions["kpi_id"] == "K034"
    ]
)

# Inspect records for this KPI
k034 = diagnostic.loc[
    diagnostic["kpi_id"] == "K034",
    [
        "employee_id",
        "reporting_month",
        "target_value",
        "actual_value",
        "better_direction",
        "achievement_pct",
        "recalculated_score",
        "score_difference",
    ],
].copy()

display(k034.head(15))

print("K034 record count:", len(k034))
print("K034 mean score difference:", k034["score_difference"].mean())
print("K034 maximum score difference:", k034["score_difference"].max())


,kpi_id,department_name,kpi_name,kpi_category,unit,better_direction,default_target,measurement_frequency
33,K034,Operations,Cost per transaction,Efficiency,currency_usd,lower,2.5,Monthly


,employee_id,reporting_month,target_value,actual_value,better_direction,achievement_pct,recalculated_score,score_difference
123,E0003,2025-01,2.31,1.88,lower,122.64,122.87,0.23
128,E0003,2025-02,2.31,2.02,lower,114.57,114.36,0.21
133,E0003,2025-03,2.31,2.15,lower,107.61,107.44,0.17
138,E0003,2025-04,2.31,2.01,lower,114.93,114.93,0.00
143,E0003,2025-05,2.31,2.12,lower,109.08,108.96,0.12
148,E0003,2025-06,2.31,2.48,lower,93.20,93.15,0.05
153,E0003,2025-07,2.31,2.02,lower,114.54,114.36,0.18
158,E0003,2025-08,2.31,1.90,lower,121.48,121.58,0.10
163,E0003,2025-09,2.31,1.96,lower,117.86,117.86,0.00
168,E0003,2025-10,2.31,2.09,lower,110.72,110.53,0.19


K034 record count: 878
K034 mean score difference: 0.10236902050113891
K034 maximum score difference: 0.4299999999999784


In [41]:

# Create a curated copy; preserve the original source data

performance_clean = performance_records.copy()

higher = performance_clean["better_direction"] == "higher"
lower = performance_clean["better_direction"] == "lower"

score = pd.Series(0.0, index=performance_clean.index)

# Higher values are better
score.loc[higher] = (
    performance_clean.loc[higher, "actual_value"]
    / performance_clean.loc[higher, "target_value"]
    * 100
)

# Lower values are better
nonzero_lower = lower & (performance_clean["actual_value"] > 0)
zero_lower = lower & (performance_clean["actual_value"] == 0)

score.loc[nonzero_lower] = (
    performance_clean.loc[nonzero_lower, "target_value"]
    / performance_clean.loc[nonzero_lower, "actual_value"]
    * 100
)

# Documented demo rule for zero actual values
score.loc[zero_lower] = 150

# Cap and round the score consistently
performance_clean["achievement_pct"] = (
    score.clip(upper=150).round(2)
)

# Rebuild status from the corrected score
performance_clean["performance_status"] = "Below Target"

performance_clean.loc[
    performance_clean["achievement_pct"] >= 80,
    "performance_status"
] = "Watch"

performance_clean.loc[
    performance_clean["achievement_pct"] >= 95,
    "performance_status"
] = "On Target"

# Validate the new table
print("Original rows:", len(performance_records))
print("Curated rows:", len(performance_clean))
print(
    "Missing scores:",
    performance_clean["achievement_pct"].isna().sum()
)
print("\nUpdated performance status counts:")
display(performance_clean["performance_status"].value_counts())

# Compare old and new scores
changed = (
    performance_clean["achievement_pct"]
    != performance_records["achievement_pct"]
)

print("\nScores updated:", int(changed.sum()))

display(
    performance_clean.loc[
        changed,
        [
            "employee_id",
            "kpi_id",
            "reporting_month",
            "target_value",
            "actual_value",
            "achievement_pct",
            "performance_status",
        ],
    ].head(10)
)


Original rows: 40250
Curated rows: 40250
Missing scores: 0

Updated performance status counts:


performance_status
On Target       25126
Watch           11169
Below Target     3955
Name: count, dtype: int64


Scores updated: 18445


,employee_id,kpi_id,reporting_month,target_value,actual_value,achievement_pct,performance_status
1,E0001,K017,2025-01,77.41,43.70,56.45,Below Target
4,E0001,K020,2025-01,19.48,31.68,61.49,Below Target
7,E0001,K018,2025-02,76.56,51.80,67.66,Below Target
11,E0001,K017,2025-03,77.41,51.63,66.70,Below Target
14,E0001,K020,2025-03,19.48,28.21,69.05,Below Target
17,E0001,K018,2025-04,76.56,57.50,75.10,Below Target
18,E0001,K019,2025-04,92.01,66.02,71.75,Below Target
24,E0001,K020,2025-05,19.48,30.00,64.93,Below Target
26,E0001,K017,2025-06,77.41,77.26,99.81,On Target
31,E0001,K017,2025-07,77.41,57.91,74.81,Below Target


In [43]:
output_path = DATA_DIR / "performance_records_curated.csv"

performance_clean.to_csv(output_path, index=False)

print("Saved to:", output_path)
print("Rows saved:", len(performance_clean))

Saved to: data\performance_records_curated.csv
Rows saved: 40250


In [45]:
# Recalculate scores independently from the curated output

check_df = pd.read_csv(
    DATA_DIR / "performance_records_curated.csv"
)

higher = check_df["better_direction"] == "higher"
lower = check_df["better_direction"] == "lower"

expected = pd.Series(0.0, index=check_df.index)

expected.loc[higher] = (
    check_df.loc[higher, "actual_value"]
    / check_df.loc[higher, "target_value"]
    * 100
)

nonzero_lower = lower & (check_df["actual_value"] > 0)
zero_lower = lower & (check_df["actual_value"] == 0)

expected.loc[nonzero_lower] = (
    check_df.loc[nonzero_lower, "target_value"]
    / check_df.loc[nonzero_lower, "actual_value"]
    * 100
)

expected.loc[zero_lower] = 150

expected = expected.clip(upper=150).round(2)

differences = (
    expected - check_df["achievement_pct"]
).abs()

print("Rows checked:", len(check_df))
print("Score mismatches:", int((differences > 0.001).sum()))
print("Duplicate records:", int(
    check_df.duplicated(
        ["employee_id", "kpi_id", "reporting_month"]
    ).sum()
))

Rows checked: 40250
Score mismatches: 0
Duplicate records: 0


SyntaxError: invalid syntax (1185458606.py, line 1)